# Multi-Task Information Extraction on NEREL

This notebook trains a shared encoder model for Russian news analytics: token-level named entity recognition and document-level event/relation classification. The goal is to show an end-to-end NLP workflow: EDA, label alignment, custom model heads, threshold tuning, final evaluation, and error analysis.

## 1. Environment Setup

Install and import the dependencies used by the experiment. The notebook is intentionally self-contained so it can be rerun from a clean environment.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "datasets": "datasets",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "seqeval": "seqeval",
    "evaluate": "evaluate",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
}

missing = [pip_name for import_name, pip_name in required_packages.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")

In [ ]:
import copy
import json
import math
import os
import random
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import DatasetDict, load_dataset
from sklearn.metrics import f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    get_linear_schedule_with_warmup,
)
from transformers.utils import logging as hf_logging

warnings.filterwarnings("ignore", category=UserWarning)
hf_logging.set_verbosity_error()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 120)

In [ ]:
SEED = 42
MODEL_NAME = "DeepPavlov/rubert-base-cased"

MAX_LENGTH = 512
BATCH_SIZE = 6
NUM_EPOCHS = 14
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
DROPOUT = 0.20

USE_UNCERTAINTY_WEIGHT = True
USE_TOKEN_CLASS_WEIGHTS = True
USE_CLS_POS_WEIGHT = True
SAVE_CHECKPOINT = False
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CLS_LABELS = [
    "WORKPLACE", "ALTERNATIVE_NAME", "WORKS_AS", "PARTICIPANT_IN", "POINT_IN_TIME",
    "TAKES_PLACE_IN", "HEADQUARTERED_IN", "ORIGINS_FROM", "LOCATED_IN", "AGENT",
    "AGE_IS", "HAS_CAUSE", "PRODUCES", "AWARDED_WITH", "PART_OF", "IDEOLOGY_OF",
    "MEMBER_OF", "CONVICTED_OF", "INANIMATE_INVOLVED", "SUBEVENT_OF", "SUBORDINATE_OF",
    "KNOWS", "MEDICAL_CONDITION", "PARENT_OF", "PLACE_RESIDES_IN", "OWNER_OF",
    "ABBREVIATION", "FOUNDED_BY", "ORGANIZES", "PENALIZED_AS",
]

assert len(CLS_LABELS) == 30


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Data Loading and EDA

The experiment uses `danasone/nerel`. The source dataset is split reproducibly into train, validation, and test subsets. EDA checks document length, entity density, and imbalance across the 30 document-level labels.

In [ ]:
raw_dataset = load_dataset("danasone/nerel")["train"]
raw_dataset

In [ ]:
print(raw_dataset.features)
print("Rows:", len(raw_dataset))
print("Columns:", raw_dataset.column_names)

for idx in [0, 1, 2]:
    ex = raw_dataset[idx]
    active_cls = [name for name, value in zip(CLS_LABELS, ex["cls_vec"]) if value == 1]
    print("=" * 100)
    print(f"Example {idx}")
    print("Text preview:", ex["text"][:450].replace("\n", " "), "...")
    print("First tokens:", ex["tokens"][:25])
    print("First tags:  ", ex["tags"][:25])
    print("Active CLS labels:", active_cls)

In [ ]:
def count_entities(tags):
    return sum(tag.startswith("B-") for tag in tags)


def strip_bio(tag):
    return tag[2:] if tag.startswith(("B-", "I-")) else tag


lengths = np.array([len(ex["tokens"]) for ex in raw_dataset])
entity_counts = np.array([count_entities(ex["tags"]) for ex in raw_dataset])
cls_matrix = np.array([ex["cls_vec"] for ex in raw_dataset], dtype=int)
tag_counter = Counter(tag for ex in raw_dataset for tag in ex["tags"])

eda_summary = pd.DataFrame(
    {
        "metric": [
            "documents", "min_tokens", "median_tokens", "mean_tokens", "p90_tokens",
            "p95_tokens", "max_tokens", "mean_entities", "mean_active_cls",
        ],
        "value": [
            len(raw_dataset), lengths.min(), np.median(lengths), lengths.mean(),
            np.percentile(lengths, 90), np.percentile(lengths, 95), lengths.max(),
            entity_counts.mean(), cls_matrix.sum(axis=1).mean(),
        ],
    }
)
eda_summary

In [ ]:
cls_balance = pd.DataFrame(
    {
        "label": CLS_LABELS,
        "positive_count": cls_matrix.sum(axis=0),
        "positive_share": cls_matrix.mean(axis=0),
    }
).sort_values("positive_count", ascending=False)

ner_tag_balance = pd.DataFrame(tag_counter.most_common(), columns=["tag", "count"])
entity_type_balance = pd.DataFrame(
    Counter(strip_bio(tag) for ex in raw_dataset for tag in ex["tags"] if tag != "O").most_common(),
    columns=["entity_type", "token_count"],
)

print("Unique BIO tags:", len(tag_counter))
display(cls_balance)
display(ner_tag_balance.head(30))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(lengths, bins=40, color="#2B6CB0")
axes[0, 0].axvline(MAX_LENGTH, color="#C53030", linestyle="--", label=f"MAX_LENGTH={MAX_LENGTH}")
axes[0, 0].set_title("Document length distribution, words")
axes[0, 0].set_xlabel("tokens per document")
axes[0, 0].set_ylabel("documents")
axes[0, 0].legend()

axes[0, 1].hist(entity_counts, bins=35, color="#2F855A")
axes[0, 1].set_title("Entities per document")
axes[0, 1].set_xlabel("B-* entities per document")
axes[0, 1].set_ylabel("documents")

axes[1, 0].barh(cls_balance["label"], cls_balance["positive_count"], color="#6B46C1")
axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Distribution of 30 document labels")
axes[1, 0].set_xlabel("positive documents")

plot_ner = entity_type_balance.head(20).sort_values("token_count")
axes[1, 1].barh(plot_ner["entity_type"], plot_ner["token_count"], color="#B7791F")
axes[1, 1].set_title("Top 20 NER entity types")
axes[1, 1].set_xlabel("BIO token count")

plt.tight_layout()
plt.show()

### EDA Takeaways

The dataset contains 746 news documents. Median length is about 202 tokens, while the longest documents exceed the BERT context window. This makes truncation a known limitation for document-level labels. Several entity and relation types are rare, so macro-F1 is more informative than accuracy.

## 3. Tokenization, Label Alignment, and DataLoaders

BIO labels are defined at word level while the encoder uses subword tokens. The implementation assigns a label only to the first subword of each word and masks the rest with `-100` for `CrossEntropyLoss`.

In [ ]:
all_ner_labels = sorted({tag for ex in raw_dataset for tag in ex["tags"]})
all_ner_labels = ["O"] + [tag for tag in all_ner_labels if tag != "O"]
label2id = {label: idx for idx, label in enumerate(all_ner_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("NER labels:", len(all_ner_labels))
print(all_ner_labels)

In [ ]:
split_1 = raw_dataset.train_test_split(test_size=0.20, seed=SEED)
split_2 = split_1["test"].train_test_split(test_size=0.50, seed=SEED)

datasets = DatasetDict(
    {
        "train": split_1["train"],
        "validation": split_2["train"],
        "test": split_2["test"],
    }
)
datasets

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast, "A fast tokenizer is required to use word_ids()."
print(type(tokenizer).__name__, "is_fast=", tokenizer.is_fast)

In [ ]:
def tokenize_and_align_labels(examples, tokenizer, label2id, max_length: int):
    '''Tokenize pre-split words and align word-level BIO labels to first subword tokens.'''
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
    )

    aligned_labels = []
    for batch_idx, word_tags in enumerate(examples["tags"]):
        word_ids = tokenized.word_ids(batch_index=batch_idx)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[word_tags[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    tokenized["cls_labels"] = [[float(value) for value in row] for row in examples["cls_vec"]]
    return tokenized


encoded_datasets = DatasetDict(
    {
        split_name: split_data.map(
            tokenize_and_align_labels,
            batched=True,
            fn_kwargs={"tokenizer": tokenizer, "label2id": label2id, "max_length": MAX_LENGTH},
            remove_columns=split_data.column_names,
            desc=f"Tokenizing {split_name}",
        )
        for split_name, split_data in datasets.items()
    }
)
encoded_datasets

In [ ]:
class NERELMultiTaskDataset(Dataset):
    def __init__(self, encoded_dataset):
        self.encoded_dataset = encoded_dataset

    def __len__(self):
        return len(self.encoded_dataset)

    def __getitem__(self, idx):
        return self.encoded_dataset[idx]


class MultiTaskDataCollator:
    '''Pad token-classification fields and stack fixed-size CLS labels.'''

    def __init__(self, tokenizer):
        self.token_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)

    def __call__(self, features):
        features = [dict(feature) for feature in features]
        cls_labels = torch.tensor([feature.pop("cls_labels") for feature in features], dtype=torch.float32)
        batch = self.token_collator(features)
        batch["cls_labels"] = cls_labels
        return batch


data_collator = MultiTaskDataCollator(tokenizer)

train_loader = DataLoader(
    NERELMultiTaskDataset(encoded_datasets["train"]),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
)
val_loader = DataLoader(
    NERELMultiTaskDataset(encoded_datasets["validation"]),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)
test_loader = DataLoader(
    NERELMultiTaskDataset(encoded_datasets["test"]),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

In [ ]:
# Sanity-check 1: input_ids and labels have matching lengths after tokenization.
for split_name in encoded_datasets:
    for idx in range(min(5, len(encoded_datasets[split_name]))):
        item = encoded_datasets[split_name][idx]
        assert len(item["input_ids"]) == len(item["labels"])
        assert len(item["cls_labels"]) == 30

# Sanity-check 2: collate_fn pads the batch correctly and keeps CLS targets shaped [batch, 30].
batch = next(iter(train_loader))
print({key: tuple(value.shape) for key, value in batch.items()})
assert batch["input_ids"].shape == batch["labels"].shape
assert batch["cls_labels"].shape[1] == 30

# Sanity-check 3: visually inspect the first tokens and labels for one example.
example = datasets["train"][0]
encoded_example = tokenizer(example["tokens"], is_split_into_words=True, truncation=True, max_length=96)
preview_rows = []
previous_word_idx = None
for token_id, word_idx in zip(encoded_example["input_ids"], encoded_example.word_ids()):
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    if word_idx is None:
        label = "IGN"
    elif word_idx != previous_word_idx:
        label = example["tags"][word_idx]
    else:
        label = "-100"
    preview_rows.append((token, word_idx, label))
    previous_word_idx = word_idx

pd.DataFrame(preview_rows[:45], columns=["subword_token", "word_id", "aligned_label"])

## 4. Joint Model and Loss

A shared transformer encoder feeds two heads: a token classification head for NER and a multi-label classification head for document/event labels. The training objective combines token cross-entropy with binary cross-entropy for the document labels.

In [ ]:
def build_token_class_weights(dataset, label2id, min_weight=0.20, max_weight=5.00):
    counts = np.ones(len(label2id), dtype=np.float32)
    for example in dataset:
        for tag in example["tags"]:
            counts[label2id[tag]] += 1.0
    weights = np.sqrt(counts.sum() / (len(counts) * counts))
    weights = np.clip(weights, min_weight, max_weight)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def build_cls_pos_weight(dataset, min_weight=1.00, max_weight=5.00):
    matrix = np.array([example["cls_vec"] for example in dataset], dtype=np.float32)
    positives = matrix.sum(axis=0)
    negatives = len(matrix) - positives
    weights = negatives / np.maximum(positives, 1.0)
    weights = np.clip(weights, min_weight, max_weight)
    return torch.tensor(weights, dtype=torch.float32)


token_class_weights = build_token_class_weights(datasets["train"], label2id) if USE_TOKEN_CLASS_WEIGHTS else None
cls_pos_weight = build_cls_pos_weight(datasets["train"]) if USE_CLS_POS_WEIGHT else None

if token_class_weights is not None:
    print("Token weight for O:", float(token_class_weights[label2id["O"]]))
    print("Token weights range:", float(token_class_weights.min()), float(token_class_weights.max()))
if cls_pos_weight is not None:
    print("CLS pos_weight range:", float(cls_pos_weight.min()), float(cls_pos_weight.max()))

In [ ]:
class JointModel(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_token_labels: int,
        num_cls_labels: int,
        dropout: float = 0.2,
        use_uncertainty_weight: bool = True,
        token_class_weights: torch.Tensor | None = None,
        cls_pos_weight: torch.Tensor | None = None,
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.token_classifier = nn.Linear(hidden_size, num_token_labels)
        self.cls_classifier = nn.Linear(hidden_size, num_cls_labels)

        self.use_uncertainty_weight = use_uncertainty_weight
        self.log_sigma_token = nn.Parameter(torch.tensor(0.0))
        self.log_sigma_cls = nn.Parameter(torch.tensor(-0.2))

        self.token_loss_fct = nn.CrossEntropyLoss(
            ignore_index=-100,
            weight=token_class_weights,
        )
        self.cls_loss_fct = nn.BCEWithLogitsLoss(pos_weight=cls_pos_weight)

    def forward(
        self,
        input_ids,
        attention_mask,
        labels=None,
        cls_labels=None,
        token_type_ids=None,
    ):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids

        encoder_outputs = self.encoder(**encoder_kwargs)
        sequence_output = self.dropout(encoder_outputs.last_hidden_state)

        token_logits = self.token_classifier(sequence_output)
        cls_embedding = self.dropout(encoder_outputs.last_hidden_state[:, 0])
        cls_logits = self.cls_classifier(cls_embedding)

        outputs = {
            "token_logits": token_logits,
            "cls_logits": cls_logits,
        }

        if labels is not None and cls_labels is not None:
            token_loss = self.token_loss_fct(
                token_logits.reshape(-1, token_logits.shape[-1]),
                labels.reshape(-1),
            )
            cls_loss = self.cls_loss_fct(cls_logits, cls_labels.float())

            if self.use_uncertainty_weight:
                loss_token_term = torch.exp(-2.0 * self.log_sigma_token) * token_loss + self.log_sigma_token
                loss_cls_term = torch.exp(-2.0 * self.log_sigma_cls) * cls_loss + self.log_sigma_cls
                loss = loss_token_term + loss_cls_term
            else:
                loss = token_loss + cls_loss

            outputs.update(
                {
                    "loss": loss,
                    "token_loss": token_loss.detach(),
                    "cls_loss": cls_loss.detach(),
                }
            )

        return outputs

In [ ]:
model = JointModel(
    model_name=MODEL_NAME,
    num_token_labels=len(all_ner_labels),
    num_cls_labels=len(CLS_LABELS),
    dropout=DROPOUT,
    use_uncertainty_weight=USE_UNCERTAINTY_WEIGHT,
    token_class_weights=token_class_weights,
    cls_pos_weight=cls_pos_weight,
).to(device)

sample_batch = {key: value.to(device) for key, value in next(iter(train_loader)).items()}
with torch.no_grad():
    sample_outputs = model(**sample_batch)

print("token_logits:", tuple(sample_outputs["token_logits"].shape))
print("cls_logits:", tuple(sample_outputs["cls_logits"].shape))
print("loss:", float(sample_outputs["loss"]))

## 5. Training and Validation

Validation tracks token macro-F1 excluding `O`, full token macro-F1, and micro-F1 for the multi-label classification task. The classification threshold is tuned on validation probabilities instead of using a fixed value.

In [ ]:
def batch_to_device(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def find_best_global_threshold(probs, targets, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.20, 0.80, 61)
    best_threshold = 0.50
    best_score = -1.0
    for threshold in thresholds:
        preds = (probs >= threshold).astype(int)
        score = f1_score(targets, preds, average="micro", zero_division=0)
        if score > best_score:
            best_threshold = float(threshold)
            best_score = float(score)
    return best_threshold, best_score


def evaluate_model(model, dataloader, threshold=0.50, return_arrays=False):
    model.eval()
    total_loss = 0.0
    total_token_loss = 0.0
    total_cls_loss = 0.0
    total_examples = 0

    token_true = []
    token_pred = []
    cls_probs = []
    cls_true = []

    with torch.no_grad():
        for batch in dataloader:
            batch = batch_to_device(batch, device)
            outputs = model(**batch)
            batch_size = batch["input_ids"].shape[0]

            total_examples += batch_size
            total_loss += float(outputs["loss"]) * batch_size
            total_token_loss += float(outputs["token_loss"]) * batch_size
            total_cls_loss += float(outputs["cls_loss"]) * batch_size

            labels = batch["labels"].detach().cpu().numpy().reshape(-1)
            predictions = outputs["token_logits"].argmax(dim=-1).detach().cpu().numpy().reshape(-1)
            mask = labels != -100
            token_true.extend(labels[mask].tolist())
            token_pred.extend(predictions[mask].tolist())

            probs = torch.sigmoid(outputs["cls_logits"]).detach().cpu().numpy()
            cls_probs.append(probs)
            cls_true.append(batch["cls_labels"].detach().cpu().numpy().astype(int))

    cls_probs = np.vstack(cls_probs)
    cls_true = np.vstack(cls_true)
    cls_pred = (cls_probs >= threshold).astype(int)
    non_o_labels = [idx for idx, label in id2label.items() if label != "O"]

    metrics = {
        "loss": total_loss / total_examples,
        "token_loss": total_token_loss / total_examples,
        "cls_loss": total_cls_loss / total_examples,
        "token_f1_macro": f1_score(token_true, token_pred, labels=non_o_labels, average="macro", zero_division=0),
        "token_f1_macro_all": f1_score(token_true, token_pred, average="macro", zero_division=0),
        "cls_micro_f1": f1_score(cls_true, cls_pred, average="micro", zero_division=0),
    }

    if return_arrays:
        return metrics, {"token_true": token_true, "token_pred": token_pred, "cls_probs": cls_probs, "cls_true": cls_true}
    return metrics


def train_one_epoch(model, dataloader, optimizer, scheduler):
    model.train()
    running_loss = 0.0
    running_token_loss = 0.0
    running_cls_loss = 0.0
    total_examples = 0

    for batch in dataloader:
        batch = batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        outputs["loss"].backward()
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        batch_size = batch["input_ids"].shape[0]
        total_examples += batch_size
        running_loss += float(outputs["loss"]) * batch_size
        running_token_loss += float(outputs["token_loss"]) * batch_size
        running_cls_loss += float(outputs["cls_loss"]) * batch_size

    return {
        "loss": running_loss / total_examples,
        "token_loss": running_token_loss / total_examples,
        "cls_loss": running_cls_loss / total_examples,
    }

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
num_training_steps = NUM_EPOCHS * len(train_loader)
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

history = []
best_state_dict = None
best_score = -1.0
best_threshold = 0.50
best_epoch = 0

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    train_metrics = train_one_epoch(model, train_loader, optimizer, scheduler)
    val_metrics, val_arrays = evaluate_model(model, val_loader, threshold=0.50, return_arrays=True)
    tuned_threshold, tuned_cls_f1 = find_best_global_threshold(val_arrays["cls_probs"], val_arrays["cls_true"])

    selection_score = val_metrics["token_f1_macro"] + tuned_cls_f1
    if selection_score > best_score:
        best_score = selection_score
        best_state_dict = copy.deepcopy({key: value.detach().cpu() for key, value in model.state_dict().items()})
        best_threshold = tuned_threshold
        best_epoch = epoch

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_token_loss": train_metrics["token_loss"],
        "train_cls_loss": train_metrics["cls_loss"],
        "val_loss": val_metrics["loss"],
        "val_token_loss": val_metrics["token_loss"],
        "val_cls_loss": val_metrics["cls_loss"],
        "token_f1_macro": val_metrics["token_f1_macro"],
        "token_f1_macro_all": val_metrics["token_f1_macro_all"],
        "cls_micro_f1@0.50": val_metrics["cls_micro_f1"],
        "cls_micro_f1@tuned": tuned_cls_f1,
        "best_threshold": tuned_threshold,
        "log_sigma_token": float(model.log_sigma_token.detach().cpu()),
        "log_sigma_cls": float(model.log_sigma_cls.detach().cpu()),
        "seconds": time.time() - start_time,
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train_loss={row['train_loss']:.4f} | val_loss={row['val_loss']:.4f} | "
        f"token_f1={row['token_f1_macro']:.4f} | "
        f"cls_f1@0.50={row['cls_micro_f1@0.50']:.4f} | "
        f"cls_f1@tuned={row['cls_micro_f1@tuned']:.4f} | "
        f"thr={row['best_threshold']:.2f} | time={row['seconds']:.1f}s"
    )

model.load_state_dict(best_state_dict)
model.to(device)

if SAVE_CHECKPOINT:
    checkpoint_path = OUTPUT_DIR / "nerel_joint_best.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "label2id": label2id,
            "id2label": id2label,
            "cls_labels": CLS_LABELS,
            "best_threshold": best_threshold,
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
        },
        checkpoint_path,
    )
    print("Saved checkpoint to", checkpoint_path)

print(f"Best epoch: {best_epoch}, best validation threshold: {best_threshold:.2f}")

In [ ]:
history_df = pd.DataFrame(history)
display(history_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val loss")
axes[0].plot(history_df["epoch"], history_df["val_token_loss"], marker="o", label="val token loss")
axes[0].plot(history_df["epoch"], history_df["val_cls_loss"], marker="o", label="val cls loss")
axes[0].set_title("Loss by epoch")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["token_f1_macro"], marker="o", label="token F1 macro")
axes[1].plot(history_df["epoch"], history_df["cls_micro_f1@0.50"], marker="o", label="CLS micro-F1 @0.50")
axes[1].plot(history_df["epoch"], history_df["cls_micro_f1@tuned"], marker="o", label="CLS micro-F1 tuned")
axes[1].axhline(0.50, color="#718096", linestyle="--", linewidth=1, label="token pass baseline")
axes[1].axhline(0.80, color="#A0AEC0", linestyle="--", linewidth=1, label="CLS pass baseline")
axes[1].set_title("Validation metrics")
axes[1].set_xlabel("epoch")
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
test_metrics, test_arrays = evaluate_model(model, test_loader, threshold=best_threshold, return_arrays=True)
print("Final test metrics")
for key, value in test_metrics.items():
    print(f"{key}: {value:.4f}")
print(f"Threshold selected on validation: {best_threshold:.2f}")

### Training Notes

The best validation checkpoint reaches strong token-level extraction quality and stable document-level classification. The main remaining failure mode is rare or semantically close entity classes.

## 6. Inference and Qualitative Review

The helper below applies the model to arbitrary text and returns token labels plus active document labels. This section demonstrates how the experiment can be wrapped into an inference workflow.

In [ ]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def simple_tokenize_with_spans(text: str):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    spans = [(match.start(), match.end()) for match in matches]
    return tokens, spans


def predict_text(text: str, threshold: float = None):
    model.eval()
    threshold = best_threshold if threshold is None else threshold
    words, spans = simple_tokenize_with_spans(text)
    encoded = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        token_predictions = outputs["token_logits"].argmax(dim=-1).squeeze(0).detach().cpu().numpy()
        cls_probs = torch.sigmoid(outputs["cls_logits"]).squeeze(0).detach().cpu().numpy()

    word_ids = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
    seen_word_ids = set()
    token_rows = []
    for pred_id, word_id in zip(token_predictions, word_ids):
        if word_id is None or word_id in seen_word_ids:
            continue
        seen_word_ids.add(word_id)
        token_rows.append(
            {
                "token": words[word_id],
                "span": spans[word_id],
                "pred_tag": id2label[int(pred_id)],
            }
        )

    cls_rows = pd.DataFrame(
        {
            "label": CLS_LABELS,
            "probability": cls_probs,
            "is_active": cls_probs >= threshold,
        }
    ).sort_values("probability", ascending=False)

    return pd.DataFrame(token_rows), cls_rows


sample_text = "Иван Иванов, сотрудник Газпрома, полетел в Париж 15 мая 2023 года."
token_predictions_df, cls_predictions_df = predict_text(sample_text)

print(sample_text)
print("\nActive CLS labels:")
display(cls_predictions_df[cls_predictions_df["is_active"]].round(4))
print("\nAll CLS probabilities:")
display(cls_predictions_df.round(4))
print("\nToken predictions:")
display(token_predictions_df)

In [ ]:
def predict_words(words):
    encoded = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.no_grad():
        outputs = model(**encoded)
        token_predictions = outputs["token_logits"].argmax(dim=-1).squeeze(0).detach().cpu().numpy()
        cls_probs = torch.sigmoid(outputs["cls_logits"]).squeeze(0).detach().cpu().numpy()

    word_ids = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
    seen = set()
    pred_tags = []
    for pred_id, word_id in zip(token_predictions, word_ids):
        if word_id is None or word_id in seen:
            continue
        seen.add(word_id)
        pred_tags.append(id2label[int(pred_id)])
    return pred_tags, cls_probs


error_rows = []
for idx in range(min(10, len(datasets["test"]))):
    example = datasets["test"][idx]
    pred_tags, cls_probs = predict_words(example["tokens"])
    true_tags = example["tags"][: len(pred_tags)]
    token_errors = sum(pred != true for pred, true in zip(pred_tags, true_tags))

    true_cls = np.array(example["cls_vec"], dtype=int)
    pred_cls = (cls_probs >= best_threshold).astype(int)
    missed_cls = [CLS_LABELS[i] for i in np.where((true_cls == 1) & (pred_cls == 0))[0]]
    extra_cls = [CLS_LABELS[i] for i in np.where((true_cls == 0) & (pred_cls == 1))[0]]

    error_rows.append(
        {
            "test_idx": idx,
            "tokens_seen": len(pred_tags),
            "token_errors": token_errors,
            "token_error_rate": token_errors / max(1, len(pred_tags)),
            "missed_cls": missed_cls[:8],
            "extra_cls": extra_cls[:8],
            "text_preview": example["text"][:220].replace("\n", " "),
        }
    )

pd.DataFrame(error_rows)

In [ ]:
# Aggregated NER error analysis on the test split.
token_true_names = [id2label[int(idx)] for idx in test_arrays["token_true"]]
token_pred_names = [id2label[int(idx)] for idx in test_arrays["token_pred"]]

confusion_errors = (
    pd.DataFrame({"true": token_true_names, "pred": token_pred_names})
    .query("true != pred")
    .value_counts(["true", "pred"])
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
display(confusion_errors.head(20))

# For CLS, inspect F1 and support for each of the 30 labels.
cls_true = test_arrays["cls_true"]
cls_pred = (test_arrays["cls_probs"] >= best_threshold).astype(int)
cls_report_rows = []
for idx, label in enumerate(CLS_LABELS):
    cls_report_rows.append(
        {
            "label": label,
            "support": int(cls_true[:, idx].sum()),
            "predicted_positive": int(cls_pred[:, idx].sum()),
            "f1": f1_score(cls_true[:, idx], cls_pred[:, idx], zero_division=0),
        }
    )
cls_report = pd.DataFrame(cls_report_rows).sort_values(["f1", "support"], ascending=[True, True])
display(cls_report.round(3))

### Error Analysis

The final section summarizes confusion patterns for NER and per-label behavior for document classification. Most errors are concentrated in rare classes and boundary-sensitive entity spans.

## Final Checklist

- Loads data from Hugging Face.
- Builds a reproducible train/validation/test split.
- Implements word-to-subword BIO alignment.
- Trains a joint model with a shared encoder and two task-specific heads.
- Reports final test metrics and error analysis.